# Práctica guiada · Dos umbrales: precisión, recall y F1 

**Consigna:** dada una lista de clientes con su **probabilidad de abandono** y lo que **realmente pasó**, calculá precisión, recall y F1 en **dos umbrales**. Elegí uno y justificalo con el **costo** de contactar a quien no se iba (falso positivo) y de perder a quien se iba (falso negativo).

Cada equipo trabaja un **caso distinto** de NovaRetail (siete casos, siete equipos). Los datos son sintéticos y se generan desde el código; ningún resultado representa una situación real de ARCA.

Cómo se usa: corré las celdas en orden. Lo único que cambiás es el número de tu equipo en la sección 1.

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook; si ya está todo, no hace nada.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "jinja2": "jinja2"}
INSTALAR_XGBOOST = False          # opcional: ponelo en True para sumar XGBoost a la comparación de modelos
if INSTALAR_XGBOOST: REQUISITOS["xgboost"] = "xgboost"

print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")


## 0 · Preparación

Esta celda carga las librerías y define la función que genera la lista de clientes de cada caso. Es un modelo ya entrenado: lo que recibís es la salida (la probabilidad), igual que en el trabajo real, cuando otro equipo te pasa los puntajes.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

VIOLETA, VERDE, NARANJA, GRIS, ROJO = "#7C5CF6", "#10D9A0", "#F59E0B", "#8888A8", "#E5484D"
plt.rcParams.update({"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 11, "axes.titleweight": "bold"})
pd.options.display.float_format = "{:,.3f}".format


from sklearn.metrics import roc_auc_score

def crear_lista(n, tasa_base, separacion, semilla):
    """Lista de clientes puntuados: probabilidad de abandono (calibrada) y abandono real."""
    azar = np.random.default_rng(semilla)
    real = (azar.random(n) < tasa_base).astype(int)
    puntaje = azar.normal(np.where(real == 1, separacion / 2, -separacion / 2), 1.0)
    logit = np.log(tasa_base / (1 - tasa_base)) + separacion * puntaje
    return pd.DataFrame({"id_cliente": [f"L{semilla}-{i:04d}" for i in range(1, n + 1)],
                         "probabilidad": (1 / (1 + np.exp(-logit))).round(4), "abandono_real": real})

## 1 · Tu caso

Esta tabla tiene los siete casos. Cada uno tiene su **costo de contactar a quien no se iba** (falso positivo), su **costo de perder a quien se iba** (falso negativo) y los **dos umbrales** que tu equipo tiene que comparar. Cambiá `MI_CASO` por el número de tu equipo y corré la celda.

In [ ]:
CASOS = {
    1: dict(nombre="Cupón del 20 % para clientes de la web", n=800, tasa_base=0.17, separacion=1.5, costo_fp=25, costo_fn=60, umbral_a=0.5, umbral_b=0.3, semilla=101),
    2: dict(nombre="Llamada del equipo Premium", n=800, tasa_base=0.05, separacion=1.7, costo_fp=15, costo_fn=150, umbral_a=0.3, umbral_b=0.1, semilla=102),
    3: dict(nombre="Clientes del primer año", n=800, tasa_base=0.3, separacion=1.3, costo_fp=8, costo_fn=45, umbral_a=0.5, umbral_b=0.25, semilla=103),
    4: dict(nombre="Reactivación de clientes inactivos", n=800, tasa_base=0.45, separacion=1.2, costo_fp=12, costo_fn=20, umbral_a=0.6, umbral_b=0.4, semilla=104),
    5: dict(nombre="Puntos extra del programa de fidelidad", n=800, tasa_base=0.2, separacion=1.4, costo_fp=30, costo_fn=45, umbral_a=0.5, umbral_b=0.35, semilla=105),
    6: dict(nombre="Mail masivo de bajo costo", n=800, tasa_base=0.143, separacion=1.0, costo_fp=0.5, costo_fn=20, umbral_a=0.25, umbral_b=0.05, semilla=106),
    7: dict(nombre="Descuento agresivo del 40 %", n=800, tasa_base=0.14, separacion=1.6, costo_fp=60, costo_fn=40, umbral_a=0.75, umbral_b=0.5, semilla=107),
}

MI_CASO = 1          # <- el número de tu equipo, del 1 al 7

caso = CASOS[MI_CASO]
print(f"Caso {MI_CASO}: {caso['nombre']}")
print(f"Costo de un falso positivo (contactar a quien no se iba): {caso['costo_fp']}")
print(f"Costo de un falso negativo (perder a quien se iba):       {caso['costo_fn']}")
print(f"Umbrales a comparar: A = {caso['umbral_a']}  y  B = {caso['umbral_b']}")

**Qué pasa en cada caso:**

| Caso | Nombre | Situación |
| --- | --- | --- |
| 1 | Cupón del 20 % para clientes de la web | NovaRetail ofrece un cupón del 20 % a quienes tienen riesgo de abandonar. Cada cupón entregado a un cliente que no se iba cuesta margen: 25. Perder a un cliente de la web que se iba cuesta 60. |
| 2 | Llamada del equipo Premium | Un ejecutivo de cuenta llama a los clientes Premium en riesgo. Cada llamada innecesaria cuesta 15. Perder a un cliente Premium que se iba cuesta 150: compran mucho y por muchos años. |
| 3 | Clientes del primer año | Los clientes con menos de un año abandonan más: 3 de cada 10. Un mensaje de bienvenida personalizado cuesta 8 si no hacía falta. Perder a un cliente nuevo cuesta 45. |
| 4 | Reactivación de clientes inactivos | Clientes con más de 120 días sin comprar. Casi la mitad no vuelve. Un mail con oferta cuesta 12 si el cliente igual volvía. Perderlo cuesta 20, porque su compra promedio es baja. |
| 5 | Puntos extra del programa de fidelidad | Regalar puntos extra a un cliente que no se iba cuesta 30. Perder a un cliente que se iba cuesta 45. |
| 6 | Mail masivo de bajo costo | Un mail estándar cuesta 0,5 por cliente. Perder a un cliente que se iba cuesta 20. El modelo de esta lista es menos preciso que los otros. |
| 7 | Descuento agresivo del 40 % | El descuento del 40 % es muy caro: cada uno entregado a un cliente que no se iba cuesta 60. Perder al cliente que se iba cuesta 40. |

## 2 · La lista de clientes

Cada fila es un cliente con la **probabilidad** de abandono que dio el modelo y lo que **realmente pasó** (`abandono_real`: 1 si abandonó). Mirá las primeras filas y la tasa real de abandono de tu lista.

In [ ]:
lista = crear_lista(caso["n"], caso["tasa_base"], caso["separacion"], caso["semilla"])
y_real, prob = lista["abandono_real"].to_numpy(), lista["probabilidad"].to_numpy()

print(f"Clientes: {len(lista):,} · Abandonaron: {y_real.sum()} ({y_real.mean():.1%})")
lista.head(8)

## 3 · Una función que calcula todo

Esta función recibe un umbral y devuelve la matriz de confusión (VP, FP, FN, VN), la precisión, el recall, el F1 y el **costo total**:

- **Precisión** = VP / (VP + FP): de los que contactamos, cuántos se iban de verdad.
- **Recall** = VP / (VP + FN): de los que se iban, cuántos encontramos.
- **F1**: un promedio de los dos que castiga que uno sea muy bajo.
- **Costo total** = costo_fp × FP + costo_fn × FN.

In [ ]:
def evaluar_umbral(y_real, prob, umbral, costo_fp, costo_fn):
    pred = prob >= umbral
    vp = int((pred & (y_real == 1)).sum()); fp = int((pred & (y_real == 0)).sum())
    fn = int((~pred & (y_real == 1)).sum()); vn = int((~pred & (y_real == 0)).sum())
    precision = vp / (vp + fp) if vp + fp else 0.0
    recall = vp / (vp + fn) if vp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"umbral": umbral, "contactos": vp + fp, "VP": vp, "FP": fp, "FN": fn, "VN": vn,
            "precisión": precision, "recall": recall, "F1": f1, "costo total": costo_fp * fp + costo_fn * fn}

## 4 · Los dos umbrales, lado a lado

**Predicción antes de correr:** ¿con qué umbral esperás contactar a más clientes? ¿Cuál va a tener mejor precisión y cuál mejor recall?

La tabla compara el umbral A con el B. Leela columna por columna.

In [ ]:
resultado = pd.DataFrame([evaluar_umbral(y_real, prob, caso["umbral_a"], caso["costo_fp"], caso["costo_fn"]),
                          evaluar_umbral(y_real, prob, caso["umbral_b"], caso["costo_fp"], caso["costo_fn"])],
                         index=["A", "B"])
resultado.style.format({"precisión": "{:.1%}", "recall": "{:.1%}", "F1": "{:.3f}", "costo total": "{:,.1f}"})

## 5 · Mirarlo en gráficos

A la izquierda, las dos matrices de confusión. A la derecha, el costo total de cada umbral dividido en sus dos partes: lo que cuesta contactar de más (falsos positivos) y lo que cuesta perder clientes (falsos negativos).

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, etiqueta in zip(ejes[:2], ["A", "B"]):
    r = resultado.loc[etiqueta]
    ax.imshow([[r["VN"], r["FP"]], [r["FN"], r["VP"]]], cmap="Purples")
    for (i, j), (nombre, valor) in zip([(0, 0), (0, 1), (1, 0), (1, 1)], [("VN", r["VN"]), ("FP", r["FP"]), ("FN", r["FN"]), ("VP", r["VP"])]):
        ax.text(j, i, f"{nombre}\n{int(valor)}", ha="center", va="center", fontsize=13, fontweight="bold",
                color="white" if valor > max(r["VN"], r["VP"]) * 0.5 else "#2B2B2B")
    ax.set_xticks([0, 1], ["Predijo: sigue", "Predijo: abandona"]); ax.set_yticks([0, 1], ["Real: sigue", "Real: abandonó"])
    ax.grid(False); ax.set_title(f"Umbral {etiqueta} = {r['umbral']}")
parte_fp = resultado["FP"] * caso["costo_fp"]; parte_fn = resultado["FN"] * caso["costo_fn"]
ejes[2].bar(resultado.index, parte_fp, color=NARANJA, label="Por contactar de más (FP)")
ejes[2].bar(resultado.index, parte_fn, bottom=parte_fp, color=VIOLETA, label="Por perder clientes (FN)")
ejes[2].set_title("Costo total por umbral"); ejes[2].set_ylabel("Costo"); ejes[2].legend(fontsize=9)
plt.tight_layout(); plt.show()

## 6 · Control antes de decidir

Antes de elegir, tres controles que tiene que pasar cualquier resultado. Si alguno falla, hay un error en el cálculo y no hay que decidir todavía.

In [ ]:
controles = {
    "Las cuatro celdas suman el total de clientes": all(resultado.loc[e, ["VP", "FP", "FN", "VN"]].sum() == len(lista) for e in "AB"),
    "Los abandonos reales son los mismos en los dos umbrales (VP + FN)": resultado.loc["A", ["VP", "FN"]].sum() == resultado.loc["B", ["VP", "FN"]].sum() == y_real.sum(),
    "Con el umbral más bajo se contacta a más clientes": (resultado["contactos"].idxmax() == ("A" if caso["umbral_a"] < caso["umbral_b"] else "B")),
}
for descripcion, ok in controles.items():
    print("✔" if ok else "✘", descripcion)

## 7 · Tu decisión

Elegí **A** o **B** y completá `ELEGIDO`. Después corré la celda: arma la ficha de decisión con tus números. Lo que falta lo ponen ustedes: **por qué** ese umbral, con el costo de cada error, y **qué riesgo** queda.

In [ ]:
ELEGIDO = "A"          # <- A o B, según lo que decidió tu equipo

r = resultado.loc[ELEGIDO]
otro = resultado.loc["B" if ELEGIDO == "A" else "A"]
print(f"CASO {MI_CASO} · {caso['nombre']}")
print(f"Decisión: usar el umbral {ELEGIDO} = {r['umbral']}")
print(f"Contactamos a {int(r['contactos'])} clientes y encontramos {int(r['VP'])} de los {int(y_real.sum())} que se iban (recall {r['recall']:.1%}, precisión {r['precisión']:.1%}).")
print(f"Costo total: {r['costo total']:,.1f} contra {otro['costo total']:,.1f} del otro umbral.")
print(f"F1: {r['F1']:.3f} contra {otro['F1']:.3f} del otro umbral.")
print("Falta: ¿por qué ese umbral? ¿qué riesgo queda?")

## 8 · Después de decidir: todos los umbrales

Solo cuando ya decidieron: este gráfico muestra qué pasaría con **cualquier** umbral. Sirve para ver si los dos que les tocaron estaban cerca de lo mejor, y si el umbral de menor costo y el de mayor F1 coinciden.

In [ ]:
barrido = pd.DataFrame([evaluar_umbral(y_real, prob, t, caso["costo_fp"], caso["costo_fn"]) for t in np.round(np.arange(0.02, 0.95, 0.01), 2)])
t_costo = barrido.loc[barrido["costo total"].idxmin(), "umbral"]; t_f1 = barrido.loc[barrido["F1"].idxmax(), "umbral"]

fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
ejes[0].plot(barrido["umbral"], barrido["costo total"], color=VIOLETA, lw=2.5); ejes[0].axvline(t_costo, color=VERDE, ls="--", label=f"Menor costo: {t_costo}")
ejes[0].set_title("Costo total por umbral"); ejes[0].set_xlabel("Umbral"); ejes[0].legend()
ejes[1].plot(barrido["umbral"], barrido["F1"], color=NARANJA, lw=2.5); ejes[1].axvline(t_f1, color=VERDE, ls="--", label=f"Mayor F1: {t_f1}")
ejes[1].set_title("F1 por umbral"); ejes[1].set_xlabel("Umbral"); ejes[1].legend()
for ax in ejes:
    for t in (caso["umbral_a"], caso["umbral_b"]): ax.axvline(t, color=GRIS, ls=":")
plt.tight_layout(); plt.show()
print(f"Umbral de menor costo: {t_costo} · Umbral de mayor F1: {t_f1} · {'Coinciden' if t_costo == t_f1 else 'NO coinciden: el F1 y el costo recomiendan cosas distintas'}")